In [ ]:
#import packages/ loading data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df1 = pd.read_csv("Freeagents2026.csv")
df2 = pd.read_csv("Catchandshoot.csv")
df3 = pd.read_csv('Per36defensivestats.csv')
df4 = pd.read_csv('defensivetrackingstats.csv')

In [ ]:
#Cleaning Free Agent data
FA = df1
FA.columns = [
    "Player", "Position", "Age", "Experience", "Team",
    "Salary", "FA_Status"
]
FA['Salary'] = FA['Salary'].astype('string')
FA['Salary'] = FA['Salary'].str.replace('$','', regex=False)
FA['Salary'] = FA['Salary'].str.replace(',','', regex=False)
FA['Salary'] = FA['Salary'].astype(float)
FA["Position"] = FA["Position"].str.upper()
FA = FA[FA["Salary"] <= 10_000_000]
FA = FA[~FA["FA_Status"].str.contains("Two-Way", na=False)]

FA = FA[FA["Position"].isin(["SF", "PF", "F"])]


#Function to create a column that shows the player's likely hood of getting
#signed based on free agency status

def signing_likelihood(fa_status):
    if pd.isna(fa_status):
        return "Unknown"

    status = fa_status.upper()

    if "TWO-WAY" in status:
        return "Very Low"
    if "CLUB" in status:
        return "Very Low"
    if "RFA" in status:
        return "Very Low"

    if "UFA" in status and "NON-BIRD" in status:
        return "High"
    if "UFA" in status:
        return "High"

    if "PLAYER" in status:
        return "Medium"

    if "EARLY BIRD" in status:
        return "Medium-Low"
    if "BIRD" in status:
        return "Low"

    return "Unknown"


FA["Signing_Likelihood"] = FA["FA_Status"].apply(signing_likelihood)



In [ ]:
#Cleanig Catch and Shoot Data
CS = df2
CS.rename(columns={'PLAYER': 'Player'}, inplace=True)

num = [
    "GP", "MIN", "PTS",
    "FGM", "FGA", "FG%",
    "3PM", "3PA", "3P%",
    "eFG%"
]

for col in num:
    CS[col] = pd.to_numeric(CS[col], errors="coerce")


CS["Total_Minutes"] = CS["GP"] * CS["MIN"]
CS["Total_C&S_3PA"] = CS["GP"] * CS["3PA"]
CS["Total_C&S_3PM"] = CS["GP"] * CS["3PM"]


CS = CS[
    (CS["GP"] >= 20) &
    (CS["Total_Minutes"] >= 400) &
    (CS["Total_C&S_3PA"] >= 50)
]

CS["C&S_3PA_per_36"] = (CS["3PA"] / CS["MIN"]) * 36

CS["Reliable_C&S_Shooter"] = CS["Total_C&S_3PA"] >= 75
CS["High_Volume_C&S_Shooter"] = CS["C&S_3PA_per_36"] >= 4.5


CS.loc[CS["3P%"] > 55, "3P%"] = np.nan
CS.loc[CS["eFG%"] > 80, "eFG%"] = np.nan

Keep_cols = [
    "Player", "GP", "MIN",
    "3P%", "eFG%",
    "C&S_3PA_per_36",
    "Total_C&S_3PA",
    "Reliable_C&S_Shooter",
    "High_Volume_C&S_Shooter"
]
CS = CS[Keep_cols].reset_index(drop=True)




/tmp/ipython-input-3502415580.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CS["C&S_3PA_per_36"] = (CS["3PA"] / CS["MIN"]) * 36
/tmp/ipython-input-3502415580.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CS["Reliable_C&S_Shooter"] = CS["Total_C&S_3PA"] >= 75
/tmp/ipython-input-3502415580.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.

In [ ]:
#Cleaning Defense_per36min data
P36_D = df3
DT = df4

num2= ["Age", "G", "GS", "MP", "DRB", "STL", "BLK", "PF"]

for col in num2:
    P36_D[col] = pd.to_numeric(P36_D[col], errors="coerce")


P36_D = P36_D[
    (P36_D["G"] >= 20) &
    (P36_D["MP"] >= 400)
]

P36_D["Pos"] = P36_D["Pos"].str.upper()

P36_D = P36_D[P36_D["Pos"].isin(["SF", "PF", "F"])]

P36_D = P36_D[
    [
        "Player",
        "DRB",
        "STL",
        "BLK",
        "PF"
    ]
].reset_index(drop=True)

#Cleaning Defense tracking data


num3 = [
    "GP", "MIN", "STL", "BLK",
    "DREB", "DFGM", "DFGA", "DFG%"
]


for col in num3:
    DT[col] = pd.to_numeric(DT[col], errors="coerce")

DT = DT[
    (DT["GP"] >= 20) &
    (DT["MIN"] >= 12)
]

DT["Reliable_Defensive_Sample"] = DT["DFGA"] >= 2.0

DT.loc[~DT["Reliable_Defensive_Sample"], "DFG%"] = np.nan

DT.loc[DT["DFG%"] > 80, "DFG%"] = np.nan

DT = DT[
    [
        "Player",
        "DFGM",
        "DFGA",
        "DFG%",
        "Reliable_Defensive_Sample"
    ]
].reset_index(drop=True)

DD = P36_D.merge(DT, on="Player", how="left")







/tmp/ipython-input-566629602.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P36_D["Pos"] = P36_D["Pos"].str.upper()
/tmp/ipython-input-566629602.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DT["Reliable_Defensive_Sample"] = DT["DFGA"] >= 2.0


In [ ]:
#merge data

Final = FA.merge(CS, on="Player", how="left")
Final = Final.merge(DD, on="Player", how="left")

#Reordering columns in a redable manner

Final = Final[
    [
        "Player", "Position", "Age", "Experience", "Team", "Salary",
        "FA_Status", "Signing_Likelihood", "GP", "MIN",
        "3P%", "eFG%", "C&S_3PA_per_36", "Total_C&S_3PA",
        "Reliable_C&S_Shooter", "High_Volume_C&S_Shooter",
        "DRB", "STL", "BLK", "PF", "DFGM", "DFGA", "DFG%", "Reliable_Defensive_Sample"
    ]
]



# function to flag likely plus two-way targets

def plus_two_way(row):

    offensive_flag = (
        row["Reliable_C&S_Shooter"] and
        row["High_Volume_C&S_Shooter"] and
        row["3P%"] >= 35 and
        row["eFG%"] >= 50
    )

    defensive_flag = (
        row["Reliable_Defensive_Sample"] and
        row["DFGA"] >= 2.0 and
        row["DFG%"] <= 75 and
        (row["STL"] >= 0.8 or row["BLK"] >= 0.5 or row["DRB"] >= 2.0)
    )

    if offensive_flag and defensive_flag:
        return True
    else:
        return False

Final["Plus_Two_Way_Target"] = Final.apply(plus_two_way, axis=1)

Final = Final.sort_values(
    by=["Plus_Two_Way_Target", "C&S_3PA_per_36", "DFGA"],
    ascending=[False, False, False]
).reset_index(drop=True)

Final

,Player,Position,Age,Experience,Team,Salary,FA_Status,Signing_Likelihood,GP,MIN,...,High_Volume_C&S_Shooter,DRB,STL,BLK,PF,DFGM,DFGA,DFG%,Reliable_Defensive_Sample,Plus_Two_Way_Target
0,Georges Niang,PF,32.4,10,BOS,8500000.0,UFA / Bird,High,79.0,21.5,...,True,4.8,0.6,0.3,4.3,1.7,2.4,71.9,True,True
1,Julian Champagnie,SF,24.4,3,SAS,3000000.0,CLUB / $3.0M,Very Low,82.0,23.6,...,True,4.7,1.1,0.7,2.1,2.0,3.0,69.4,True,True
2,Ziaire Williams,SF,24.2,4,BKN,6250000.0,CLUB / $6.3M,Very Low,63.0,24.5,...,True,5.3,1.4,0.7,3.5,1.5,2.3,64.6,True,True
3,Dean Wade,PF,29.0,7,CLE,6166667.0,UFA / Bird,High,59.0,21.2,...,True,5.7,1.2,0.6,2.9,1.8,3.0,61.1,True,True
4,Nicolas Batum,SF,36.9,17,LAC,5741640.0,CLUB / $5.9M,Very Low,78.0,17.5,...,True,4.5,1.4,0.9,2.9,1.2,2.1,58.1,True,True
5,Chris Boucher,PF,32.9,9,BOS,3287409.0,UFA / Non-Bird,High,50.0,17.2,...,True,6.4,1.0,1.0,2.8,1.1,1.7,NaN,False,False
6,Lindy Waters III,SF,28.3,5,SAS,2461463.0,UFA / Non-Bird,High,52.0,15.0,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7,Tyrese Martin,SF,26.8,3,BKN,1413875.0,RFA / Early Bird,Very Low,60.0,21.9,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8,Jalen Wilson,PF,25.1,3,BKN,1654511.0,RFA / Bird,Very Low,79.0,25.7,...,True,3.5,0.7,0.1,2.9,1.5,2.0,75.2,True,False
9,Larry Nance Jr.,PF,32.9,11,CLE,3634153.0,UFA / Non-Bird,High,24.0,19.3,...,True,6.2,1.6,1.0,2.9,1.4,2.2,64.2,True,False


In [ ]:
# Ranking top players to fit the role we are looking for


salary_cap = 10_000_000
realistic_likelihoods = ["High", "Medium"]

Filtered = Final[
    (Final["Salary"] <= salary_cap) &
    (Final["Signing_Likelihood"].isin(realistic_likelihoods))
]

Filtered["High_Offensive_Involvement"] = Filtered["C&S_3PA_per_36"] >= 4.5

Filtered["Defensive_Impact_per36"] = Filtered[["DRB", "STL", "BLK"]].sum(axis=1)

Filtered["Total_Impact"] = Filtered["C&S_3PA_per_36"] + Filtered["Defensive_Impact_per36"]

Filtered["Plus_Two_Way_Target"] = Filtered["Plus_Two_Way_Target"].fillna(False)


Ranked = Filtered.dropna(subset=["Plus_Two_Way_Target", "Total_Impact"]) \
    .sort_values(
        by=["Plus_Two_Way_Target", "Total_Impact"],
        ascending=[False, False]
    ).reset_index(drop=True)

Ranked[[
    "Player","Position","Age","Team","Salary",
    "Signing_Likelihood","C&S_3PA_per_36","Defensive_Impact_per36",
    "Total_Impact","Plus_Two_Way_Target"
]].head(20)





/tmp/ipython-input-3534841332.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Filtered["High_Offensive_Involvement"] = Filtered["C&S_3PA_per_36"] >= 4.5
/tmp/ipython-input-3534841332.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Filtered["Defensive_Impact_per36"] = Filtered[["DRB", "STL", "BLK"]].sum(axis=1)
/tmp/ipython-input-3534841332.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

,Player,Position,Age,Team,Salary,Signing_Likelihood,C&S_3PA_per_36,Defensive_Impact_per36,Total_Impact,Plus_Two_Way_Target
0,Georges Niang,PF,32.4,BOS,8500000.0,High,8.204651,5.7,13.904651,True
1,Dean Wade,PF,29.0,CLE,6166667.0,High,6.113208,7.5,13.613208,True
2,Chris Boucher,PF,32.9,BOS,3287409.0,High,7.325581,8.4,15.725581,False
3,Larry Nance Jr.,PF,32.9,CLE,3634153.0,High,5.782383,8.8,14.582383,False
4,Simone Fontecchio,SF,30.0,MIA,8000000.0,High,5.672727,5.9,11.572727,False
5,Kelly Oubre Jr.,SF,30.0,PHI,8182575.0,High,3.433526,6.8,10.233526,False
6,Taurean Prince,PF,31.7,MIL,3559818.0,Medium,4.383764,5.7,10.083764,False
7,Haywood Highsmith,F,29.0,BKN,5408000.0,High,4.390244,5.6,9.990244,False
8,Lindy Waters III,SF,28.3,SAS,2461463.0,High,6.960000,0.0,6.960000,False
9,Guerschon Yabusele,PF,29.9,NYK,5637500.0,Medium,4.915129,0.0,4.915129,False
